In [1]:
import functools
import time
from typing import List

import numpy as np
import jax
import jax.numpy as jnp

from jax.experimental.pallas.ops.tpu.megablox.gmm import gmm
from jax.experimental.shard_map import shard_map
from jax.sharding import Mesh, NamedSharding, PartitionSpec

/home/lsiyuan_google_com/miniconda3/envs/torch312/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [2]:
P = PartitionSpec

mesh = Mesh(jax.devices(), ('x',))

seq_p_spec = P('x',)
seq_p_sharding = NamedSharding(mesh, seq_p_spec)

In [3]:
n_tokens = 2048
dim = 7168
dtype = jnp.bfloat16

x = jnp.ones((n_tokens, dim), dtype=dtype)
x = jax.device_put(x, seq_p_sharding)

def ag(x):
    out = jax.lax.all_gather(x, 'x')
    return out

def ar(x):
    out = jax.lax.psum(x, 'x')
    return out

def rs(x):
    out = jax.lax.psum_scatter(x, 'x', tiled=True)
    return out

ag_shmapped = jax.jit(shard_map(ag, mesh=mesh, in_specs=P('x'), out_specs=P(), check_rep=False))

ar_shmapped = jax.jit(shard_map(ar, mesh=mesh, in_specs=P('x'), out_specs=P(), check_rep=False))

rs_shmapped = jax.jit(shard_map(rs, mesh=mesh, in_specs=P(), out_specs=P('x'), check_rep=False))

In [4]:
def benchmark(f, x):
    out = f(x)
    out.block_until_ready()
    
    start_time = time.perf_counter()
    n_iter = 200
    
    for i in range(n_iter):
        out = f(x)
        out.block_until_ready()
    
    end_time = time.perf_counter()
    latency_ms = (end_time - start_time) * 1000 / n_iter
    data_size_bytes = np.prod(x.shape) * x.itemsize
    bw_gb_s = data_size_bytes / latency_ms * 1000 / 1024 / 1024 / 1024
    print(f"shape {x.shape}, dtype {dtype}, datasize(bytes): {data_size_bytes}, latency {latency_ms} ms, bw {bw_gb_s} GB/s")

In [12]:
for T in [16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 8192*2, 8192*2*2, 8192*2*2*2, 8192*2*2*2*2, 8192*2*2*2*2*2, 8192*2*2*2*2*2*2]:
    x = jnp.ones((T, dim), dtype=dtype)
    x = jax.device_put(x, seq_p_sharding)
    benchmark(ag_shmapped, x)

shape (16, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 229376, latency 0.284378204960376 ms, bw 0.7511934569836858 GB/s
shape (32, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 458752, latency 0.2819555997848511 ms, bw 1.5152956496555283 GB/s
shape (64, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 917504, latency 0.2827667002566159 ms, bw 3.02189821759257 GB/s
shape (128, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 1835008, latency 0.29081215034238994 ms, bw 5.876592064629741 GB/s
shape (256, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 3670016, latency 0.29619790031574667 ms, bw 11.539476634900007 GB/s
shape (512, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 7340032, latency 0.3198463993612677 ms, bw 21.37256356066958 GB/s
shape (1024, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 14680064, latency 0.3576557000633329 ms, bw 38.22635847151049 GB/s
shape (2048, 7168), dtype <class 'j

In [13]:
for T in [16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 8192*2, 8192*2*2, 8192*2*2*2, 8192*2*2*2*2, 8192*2*2*2*2*2, 8192*2*2*2*2*2*2]:
    x = jnp.ones((T, dim), dtype=dtype)
    x = jax.device_put(x, seq_p_sharding)
    benchmark(ar_shmapped, x)

shape (16, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 229376, latency 0.2606325491797179 ms, bw 0.819633033354162 GB/s
shape (32, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 458752, latency 0.26184545014984906 ms, bw 1.631672780663156 GB/s
shape (64, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 917504, latency 0.2769041492138058 ms, bw 3.0858771525312956 GB/s
shape (128, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 1835008, latency 0.26924735051579773 ms, bw 6.347265336970243 GB/s
shape (256, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 3670016, latency 0.27069490402936935 ms, bw 12.626646084291131 GB/s
shape (512, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 7340032, latency 0.28551984927617013 ms, bw 23.942074490897877 GB/s
shape (1024, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 14680064, latency 0.2861102996394038 ms, bw 47.7853297040729 GB/s
shape (2048, 7168), dtype <class

In [5]:
for T in [16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 8192*2, 8192*2*2, 8192*2*2*2, 8192*2*2*2*2, 8192*2*2*2*2*2, 8192*2*2*2*2*2*2]:
    x = jnp.ones((T, dim), dtype=dtype)
    x = jax.device_put(x, NamedSharding(mesh, P()))
    benchmark(rs_shmapped, x)

shape (16, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 229376, latency 0.30851515009999275 ms, bw 0.6924231980366692 GB/s
shape (32, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 458752, latency 0.3011431510094553 ms, bw 1.4187475036966233 GB/s
shape (64, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 917504, latency 0.3146695997565985 ms, bw 2.715521893951504 GB/s
shape (128, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 1835008, latency 0.3116021992173046 ms, bw 5.484506782342032 GB/s
shape (256, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 3670016, latency 0.32976340502500534 ms, bw 10.364912230757751 GB/s
shape (512, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 7340032, latency 0.3643342503346503 ms, bw 18.762818740541185 GB/s
shape (1024, 7168), dtype <class 'jax.numpy.bfloat16'>, datasize(bytes): 14680064, latency 0.44278605026192963 ms, bw 30.876932531890777 GB/s
shape (2048, 7168), dtype <cla

In [10]:
x = jnp.arange(16).reshape(8, 2)

In [13]:
x = jax.device_put(x, NamedSharding(mesh, P('x', None)))

In [16]:
x.addressable_shards

[Shard(device=TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), index=(slice(0, 1, None), slice(None, None, None)), replica_id=0, data=[[0 1]]),
 Shard(device=TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), index=(slice(1, 2, None), slice(None, None, None)), replica_id=0, data=[[2 3]]),
 Shard(device=TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), index=(slice(2, 3, None), slice(None, None, None)), replica_id=0, data=[[4 5]]),
 Shard(device=TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), index=(slice(3, 4, None), slice(None, None, None)), replica_id=0, data=[[6 7]]),
 Shard(device=TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), index=(slice(4, 5, None), slice(None, None, None)), replica_id=0, data=[[8 9]]),
 Shard(device=TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), index=(slice(5, 6, None), slice(None, None, None)), replica_id=0, data=[[10 11]]),
 Shard(device=TpuDevice(id=6, pr

In [20]:
shmap_psum = shard_map(lambda x: jax.lax.psum(x, 'x'), mesh=mesh, in_specs=seq_p_spec, out_specs=P(), check_rep=False)

In [21]:
shmap_psum(x)

Array([[56, 64]], dtype=int32)

In [28]:
shmap_psum_scatter = shard_map(lambda x: jax.lax.psum_scatter(x, 'x', scatter_dimension=0, tiled=True), mesh=mesh, in_specs=seq_p_spec, out_specs=P('x'), check_rep=False)

In [29]:
shmap_psum_scatter(x)

ValueError: tiled reduce_scatter operand scatter dimension size 1 must be divisible by shard_count 8

In [ ]:
print(jnp.bfloat16)